# 8. Query Transformations

Rewrite, step-back and decompose the question before retrieval.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Users type short, vague, multi-part or conversational questions, and the embedding of a messy question is a poor search key. **Query transformation** improves the key before searching: rewrite it, abstract it (step-back), or break it into simpler sub-questions.

## How it works

1. **Rewrite** into a clean standalone query
2. **Step-back**: ask the broader question to fetch background
3. **Decompose** into sub-questions answered independently
4. Retrieve for every variant
5. Merge results keeping the best score per chunk

```text
                 +--> rewrite ----+
 messy question -+--> step-back --+--> retrieve each --> merge/dedupe --> LLM
                 +--> sub-queries +
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 12:13:35.701000 13488 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
def rewrite(q):
    return llm(f"Rewrite this as a clear, standalone search query. Reply with the query only.\n\n{q}")

def step_back(q):
    return llm(f"Write a more general question that would retrieve useful background for this one. Reply with the question only.\n\n{q}")

def decompose(q):
    return llm_json(f"Break this question into 2-3 simple sub-questions. Reply with a JSON list of strings.\n\n{q}") or [q]

original = "so how do u make the search thing better at finding exact stuff like product codes and also fuzzy meaning?"
print("rewrite  :", rewrite(original))
print("step-back:", step_back(original))
print("sub-qs   :", decompose(original))

rewrite  : "improve google search for exact matches and fuzzy queries"


step-back: What are some effective strategies for optimizing search functionality to balance precision (e.g., exact matches, product codes) with relevance (e.g., fuzzy matching, semantic understanding)?


sub-qs   : ['Optimize search algorithms', 'Implement fuzzy matching techniques', 'Use natural language processing (NLP) for product code extraction']


Retrieve with **every** variant and merge the results, keeping the best score per chunk.

In [3]:
def multi_query_search(q, k=4):
    queries = [q, rewrite(q), step_back(q)] + decompose(q)
    best = {}
    for sub in queries:
        for h in store.search(sub, k):
            if h["idx"] not in best or h["score"] > best[h["idx"]]["score"]:
                best[h["idx"]] = h
    return sorted(best.values(), key=lambda h: -h["score"])[:k]

show(multi_query_search(original))
print(answer(original, [h["text"] for h in multi_query_search(original)]))

0.477  res the query with every vector, which is accurate but slow at scale. Approximate nearest neighbour indexes such as HNSW and IVF trade a little recall for large...
0.457  the embedding. A chunk size of around 500 to 1000 characters with 10 to 20 percent overlap is a common starting point that should be tuned on real questions.  #...
0.445  uch as what the main themes of a corpus are. Plain vector search struggles with such questions because no single chunk contains the answer.  ## Agents and Agent...
0.391  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted score or with Reciprocal Rank Fusion, which adds 1 divided by...


To improve the search's ability to find exact stuff like product codes and fuzzy meaning, you can use BM25 as a lexical ranking function. It scores documents by how often query terms appear, weighted by how rare each term is across the collection. This makes it excellent at exact keywords, product codes, and names that embeddings often miss.

Additionally, you can consider using Hybrid Search, which runs both dense retrieval (using vector search) and BM25. The dense retrieval part is better at paraphrases and meaning, while BM25 excels at exact keywords. By combining the two, you can leverage the strengths of each approach to improve recall in production.

You can also try Reranking, where a cross-encoder reads the query and a candidate passage together and outputs a relevance score. This can help refine the results and provide more accurate answers.

It's worth noting that using tools like FAISS, HNSW, IVF, Milvus, Qdrant, and Weaviate as separate database services with filtering, per

## Compare the three transformations individually
Which variant finds the evidence best for a *messy* question?

In [4]:
messy = [
    "how big r the vectors from minilm thing",
    "whats the deal w/ fusing bm25 and dense, what number do u use",
    "that model that reads query+passage together, name of small one?",
]
for q in messy:
    print("ORIGINAL :", q)
    print("  rewrite  :", rewrite(q))
    print("  step-back:", step_back(q))
    print("  sub-qs   :", decompose(q), "\n")

ORIGINAL : how big r the vectors from minilm thing


  rewrite  : "Minilm vector size"


  step-back: What is Minilm, and what context does it relate to?


  sub-qs   : ['Minilm model vector size', 'How many parameters does Minilm have?', "What is the dimensionality of Minilm's vector space?"] 

ORIGINAL : whats the deal w/ fusing bm25 and dense, what number do u use


  rewrite  : "bm25 vs dense: how to fuse these two algorithms"


  step-back: What is the purpose of combining BM25 and Dense Passage Ranking algorithms in information retrieval systems?


  sub-qs   : ['BM25', 'Dense', 'Weighted Average'] 

ORIGINAL : that model that reads query+passage together, name of small one?


  rewrite  : "model that combines query and passage for named entity recognition"


  step-back: What is the name of the popular conversational AI model architecture that combines both input and context to generate human-like responses?


  sub-qs   : ['BERT', 'RoBERTa', 'DistilBERT'] 



## Does it help retrieval? Evaluate on the golden set
We compare plain search against each transformation, using the *original* golden question each time, so improvements come from the transformation itself.

In [5]:
def via(fn):
    def search(q, k=3):
        return store.search(fn(q), k)
    return search

compare_retrievers({
    "original": store.search,
    "rewrite": via(rewrite),
    "step-back": via(step_back),
    "multi-query merge": lambda q, k=3: multi_query_search(q, k),
}, k=3)

,hit@1,hit@3,MRR
method,,,
original,0.80,1.00,0.89
rewrite,0.70,0.95,0.81
step-back,0.80,0.90,0.84
multi-query merge,0.75,0.95,0.83


## Test on genuinely messy queries
Golden questions are already well-formed, so gains are small there. Rewriting matters more for the messy ones. Let us check where the right evidence ranks.

In [6]:
evidence = ["384-dimensional vectors", "1 divided by (60 plus rank)", "cross-encoder/ms-marco-MiniLM-L-6-v2 is a small"]
import pandas as pd
rows = []
for q, ev in zip(messy, evidence):
    rows.append({"query": q[:45],
                 "original rank": evidence_rank(store.search(q, 5), ev),
                 "rewritten rank": evidence_rank(store.search(rewrite(q), 5), ev),
                 "multi-query rank": evidence_rank(multi_query_search(q, 5), ev)})
pd.DataFrame(rows)

,query,original rank,rewritten rank,multi-query rank
0,how big r the vectors from minilm thing,1.0,1.0,1.0
1,"whats the deal w/ fusing bm25 and dense, what",NaN,3.0,2.0
2,"that model that reads query+passage together,",NaN,NaN,NaN


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Big gains for vague or conversational queries | Adds an LLM call (latency) per transformation |
| Sub-queries enable multi-part questions | Rewrites can drift from the user's intent |
| No re-indexing needed | Little benefit when questions are already clean |

## When to use it

Chatbots, voice or search-box input, and complex questions that bundle several asks.

## Experiments to try

- Use the chat history to rewrite follow-up questions ('and what about its price?').
- Run sub-questions in parallel.
- Add a check that the rewrite keeps all named entities.

## Key takeaways

- The query is the weakest link in many RAG systems.
- Always retrieve with the original query too.
- Merge by best score and dedupe.

---
**Next:** [9. HyDE - Hypothetical Document Embedding](../09_HyDe_Hypothetical_Document_Embedding/HyDe_Hypothetical_Document_Embedding.ipynb)